In [ ]:
import requests
import pandas as pd
import time

# =====================================================================
# 1. MAP THE CENTROIDS (Latitude & Longitude)
# =====================================================================
# You can get the exact coordinates for the rest by right-clicking on Google Maps!
barangay_centroids = {
    'Amaya': {'lat': 14.3297, 'lon': 120.8351},
    'Bagtas': {'lat': 14.3362, 'lon': 120.8585},
    'Biga': {'lat': 14.3188, 'lon': 120.8550},
    'Biwas': {'lat': 14.4031, 'lon': 120.8513},
    'Bucal': {'lat': 14.3803, 'lon': 120.8661},
    'Bunga': {'lat': 14.3438, 'lon': 120.8686},
    'Calibuyo': {'lat': 14.3591, 'lon': 120.8068},
    'Capipisa': {'lat': 14.3512, 'lon': 120.7896},
    'Daang Amaya': {'lat': 14.3914, 'lon': 120.8526},
    'Halayhay': {'lat': 14.3686, 'lon': 120.8126},
    'Julugan': {'lat': 14.3522, 'lon': 120.8443},
    'Lambingan': {'lat': 14.3451, 'lon': 120.8054},
    'Mulawin': {'lat': 14.3835, 'lon': 120.8528},
    'Paradahan': {'lat': 14.3212, 'lon': 120.8606},
    'Poblacion': {'lat': 14.3394, 'lon': 120.8545},
    'Punta': {'lat': 14.3569, 'lon': 120.8585},
    'Sahud Ulan': {'lat': 14.3715, 'lon': 120.8248},
    'Sanja Mayor': {'lat': 14.3735, 'lon': 120.8547},
    'Santol': {'lat': 14.3759, 'lon': 120.8700},
    'Tanauan': {'lat': 14.2937, 'lon': 120.8298},
    'Tres Cruses': {'lat': 14.2886, 'lon': 120.8383},
    
}

In [2]:
# =====================================================================
# 2. CROSS-REFERENCE WITH SPATIAL API
# =====================================================================
def fetch_environmental_data(barangay_name, lat, lon):
    print(f"📡 Extracting satellite grid data for {barangay_name}...")
    
    # Open-Meteo Free API Endpoint (Extracting PM10 and PM2.5 for Air Quality)
    url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&current=pm10,pm2_5&timezone=Asia%2FSingapore"
    
    try:
        response = requests.get(url)
        response.raise_for_status() # Check for errors
        data = response.json()
        
        # Extract the specific values from the API's JSON response
        current_data = data.get('current', {})
        pm10 = current_data.get('pm10', 0)
        pm2_5 = current_data.get('pm2_5', 0)
        
        # Calculate a simplified estimated AQI based on PM2.5
        # (This is a basic estimation formula; you can refine this later)
        estimated_aqi = round(pm2_5 * 3.3) 
        
        return {
            'Barangay': barangay_name,
            'Latitude': lat,
            'Longitude': lon,
            'Raw_PM10': pm10,
            'Raw_PM2.5': pm2_5,
            'Extracted_AQI': estimated_aqi
        }
        
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Error fetching data for {barangay_name}: {e}")
        return None

In [3]:
# =====================================================================
# 3. RUN THE EXTRACTION LOOP
# =====================================================================
extracted_records = []

for brgy, coords in barangay_centroids.items():
    # Fetch data for this specific coordinate
    result = fetch_environmental_data(brgy, coords['lat'], coords['lon'])
    
    if result:
        extracted_records.append(result)
        
    # Be polite to free APIs by pausing for 1 second between requests
    time.sleep(1) 

📡 Extracting satellite grid data for Amaya...
📡 Extracting satellite grid data for Bagtas...
📡 Extracting satellite grid data for Biga...
📡 Extracting satellite grid data for Biwas...
📡 Extracting satellite grid data for Bucal...
📡 Extracting satellite grid data for Bunga...
📡 Extracting satellite grid data for Calibuyo...
📡 Extracting satellite grid data for Capipisa...
📡 Extracting satellite grid data for Daang Amaya...
📡 Extracting satellite grid data for Halayhay...
📡 Extracting satellite grid data for Julugan...
📡 Extracting satellite grid data for Lambingan...
📡 Extracting satellite grid data for Mulawin...
📡 Extracting satellite grid data for Paradahan...
📡 Extracting satellite grid data for Poblacion...
📡 Extracting satellite grid data for Punta...
📡 Extracting satellite grid data for Sahud Ulan...
📡 Extracting satellite grid data for Sanja Mayor...
📡 Extracting satellite grid data for Santol...
📡 Extracting satellite grid data for Tanauan...
📡 Extracting satellite grid data fo

In [4]:
# =====================================================================
# 4. CONVERT TO PANDAS DATAFRAME & SAVE
# =====================================================================
print("\n✅ Extraction Complete! Formatting data...")

df = pd.DataFrame(extracted_records)

# Display the beautiful dataset in your terminal
print("\n--- Interpolated Environmental Data ---")
print(df.to_string(index=False))

# Save it directly to a CSV file so your XGBoost model can train on it
df.to_csv("tanza_barangay_environment_data.csv", index=False)
print("\n💾 Saved successfully to 'tanza_barangay_environment_data.csv'")


✅ Extraction Complete! Formatting data...

--- Interpolated Environmental Data ---
   Barangay  Latitude  Longitude  Raw_PM10  Raw_PM2.5  Extracted_AQI
      Amaya   14.3297   120.8351      25.5       21.8             72
     Bagtas   14.3362   120.8585      25.5       21.8             72
       Biga   14.3188   120.8550      25.5       21.8             72
      Biwas   14.0310   120.8513      14.3       11.3             37
      Bucal   14.3803   120.8661      25.5       21.8             72
      Bunga   14.3438   120.8686      25.5       21.8             72
   Calibuyo   14.3591   120.8068      25.5       21.8             72
   Capipisa   14.3512   120.7896      25.5       21.8             72
Daang Amaya   14.3914   120.8526      25.5       21.8             72
   Halayhay   14.3686   120.8126      25.5       21.8             72
    Julugan   14.3522   120.8443      25.5       21.8             72
  Lambingan   14.3451   120.8054      25.5       21.8             72
    Mulawin   14.38